# Smart Shuttle Passenger Counting - Fixed Training Notebook
This version is updated to:
- diagnose the `torch` / `ultralytics` import problem
- help you remove local file conflicts like `torch.py`
- reinstall PyTorch cleanly
- train with more accuracy-focused settings
- compare models safely on an 8GB GPU

**Important:** after the reinstall cell finishes, restart the kernel before running the import cell.


In [ ]:
# Cell 1 - Check Python version, working folder, and possible file conflicts
import os
import sys
from pathlib import Path

print('Python version:', sys.version)
print('Current working directory:', os.getcwd())
print('
Files in current folder:')
for name in sorted(os.listdir('.')):
    print('-', name)

conflicts = []
for bad_name in ['torch.py', 'torch', 'ultralytics.py', 'matplotlib.py']:
    if os.path.exists(bad_name):
        conflicts.append(bad_name)

print('
Possible local conflicts:', conflicts if conflicts else 'None found')

if conflicts:
    print('
Rename these files/folders before continuing, then restart the kernel.')


In [ ]:
# Cell 2 - Clean reinstall of core packages
# Run this ONCE. After it finishes, restart the kernel.
%pip uninstall -y torch torchvision torchaudio ultralytics matplotlib
%pip install --upgrade pip setuptools wheel
%pip install matplotlib pandas jupyter
%pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu128
%pip install ultralytics


## Restart kernel here
After Cell 2 finishes:
1. Click **Kernel -> Restart Kernel**
2. Then run the next cells from top to bottom


In [ ]:
# Cell 3 - Test imports after restart
import os
import sys
import torch
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Image, display
from ultralytics import YOLO

print('Python:', sys.version)
print('Torch version:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU name:', torch.cuda.get_device_name(0))
print('Pandas:', pd.__version__)
print('Matplotlib:', plt.matplotlib.__version__)
print('Ultralytics import OK')


In [ ]:
# Cell 4 - Training configuration (accuracy-focused but still safe for 8GB VRAM)
import os

# Change this only if your dataset path is different
data_yaml_path = 'datasets/smart_bus_final/data.yaml'

if not os.path.exists(data_yaml_path):
    raise FileNotFoundError(f'Dataset YAML not found: {data_yaml_path}')

# Better accuracy than nano models, but still realistic for an 8GB laptop GPU
models_to_test = {
    'YOLOv8s': 'yolov8s.pt',
    'YOLOv10s': 'yolov10s.pt',
    'YOLO11s': 'yolo11s.pt'
}

# If your GPU memory is okay later, you can try medium models one by one:
# 'yolov8m.pt', 'yolov10m.pt', 'yolo11m.pt'

epochs = 100
imgsz = 640
batch = 8          # reduce to 4 if you get out-of-memory errors
patience = 20
workers = 4
project_name = 'Smart_Shuttle_Comparison_Fixed'

print('Dataset:', data_yaml_path)
print('Models:', models_to_test)
print('Epochs:', epochs)
print('Image size:', imgsz)
print('Batch:', batch)


In [ ]:
# Cell 5 - Train all models
results_dict = {}

for model_name, weights in models_to_test.items():
    print(f"\n{'='*60}\nStarting training for {model_name}\n{'='*60}")

    model = YOLO(weights)

    train_results = model.train(
        data=data_yaml_path,
        epochs=epochs,
        imgsz=imgsz,
        batch=batch,
        patience=patience,
        device=0 if torch.cuda.is_available() else 'cpu',
        workers=workers,
        cache=True,
        optimizer='AdamW',
        lr0=0.001,
        project=project_name,
        name=model_name,
        exist_ok=True,
        plots=True,
        pretrained=True,
        verbose=True
    )

    val_results = model.val(
        data=data_yaml_path,
        split='val',
        imgsz=imgsz,
        batch=batch,
        device=0 if torch.cuda.is_available() else 'cpu'
    )

    results_dict[model_name] = {
        'train_results': train_results,
        'val_results': val_results
    }

    print(f"Completed: {model_name}")


In [ ]:
# Cell 6 - Show training graphs
print('Training graphs:
')

for model_name in models_to_test.keys():
    print(f'--- {model_name} ---')
    results_img_path = f"{project_name}/{model_name}/results.png"

    if os.path.exists(results_img_path):
        display(Image(filename=results_img_path, width=900))
    else:
        print(f'Results graph not found for {model_name}: {results_img_path}')


In [ ]:
# Cell 7 - Build final comparison table
comparison_data = []

for model_name in models_to_test.keys():
    csv_path = f"{project_name}/{model_name}/results.csv"

    if os.path.exists(csv_path):
        df = pd.read_csv(csv_path)
        df.columns = df.columns.str.strip()
        best_epoch = df.iloc[-1]

        comparison_data.append({
            'Model': model_name,
            'Precision': round(float(best_epoch.get('metrics/precision(B)', 0)), 4),
            'Recall': round(float(best_epoch.get('metrics/recall(B)', 0)), 4),
            'mAP50': round(float(best_epoch.get('metrics/mAP50(B)', 0)), 4),
            'mAP50-95': round(float(best_epoch.get('metrics/mAP50-95(B)', 0)), 4)
        })
    else:
        print(f'CSV not found for {model_name}: {csv_path}')

if comparison_data:
    final_df = pd.DataFrame(comparison_data)
    final_df = final_df.sort_values(by='mAP50-95', ascending=False).reset_index(drop=True)
    print('Final model ranking:')
    display(final_df)
else:
    print('No comparison data found. Make sure training finished successfully.')


In [ ]:
# Cell 8 - Plot final comparison chart
if 'final_df' in globals() and not final_df.empty:
    plt.figure(figsize=(10, 6))
    plt.bar(final_df['Model'], final_df['mAP50-95'])
    plt.title('Passenger Counting Model Comparison (mAP50-95)')
    plt.ylabel('mAP50-95')
    plt.xlabel('Model')
    plt.ylim(0, 1.0)
    plt.show()
else:
    print('final_df is not ready yet.')


## Extra note
If you still get a `torch` import error after reinstalling, the problem is usually one of these:
- a local file or folder named `torch`
- kernel not restarted after installation
- broken virtual environment

In that case, create a fresh venv and install again there.


In [4]:
from pathlib import Path
from PIL import Image

# Get the directory where this script is located
script_dir = Path(__file__).parent.resolve()

# Option A: If script is in smart-shuttle-ai-system folder
dataset_root = script_dir / "ai_models/passenger_counting/dataset"

# Option B: If script is elsewhere, use absolute path
# dataset_root = Path("C:/your/full/path/to/smart-shuttle-ai-system/ai_models/passenger_counting/dataset")

# Only check splits that actually exist
splits = ["train", "valid", "test"]
image_exts = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

print(f"Dataset root: {dataset_root.resolve()}")
print(f"Dataset exists: {dataset_root.exists()}")

for split in splits:
    split_dir = dataset_root / split / "images"
    if not split_dir.exists():
        print(f"\n[{split.upper()}] Folder not found: {split_dir}")
        continue

    # ... rest of your code remains the same

NameError: name '__file__' is not defined